# Creacion del Modelo Entidad-Relación basada en un Archivo CSV

Este notebook construye diferentes datasets los cuales serviran para la creacion de un modelo entidad-relación y esta basado en los datos del archivo de ventas. `ventasPorTransaccion.csv`, el cual es un periodo de 20250101-20250831.

**Objetivo:** Crear datasets separados para stores, supervisors, products, brands, laboratories, categories y types, cada una con su propio ID único, y modifica el dataset principal con ID's para establecer las relaciones apropiadas con los datasets genreados.

## 1. Importacion de Librerías:

Importamos pandas, numpy y otras librerías necesarias para la manipulación y análisis de datos.

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("Librerías importadas exitosamente")
print(f"Pandas versión: {pd.__version__}")
print(f"Numpy versión: {np.__version__}")

## 2. Carga y Exploracion del CSV.

Cargamos el archivo CSV usando pandas y exploramos su estructura, columnas y tipos de datos.

In [ ]:
# Cargar el archivo CSV
csv_file_path = 'ventasPorTransaccion.csv'

try:
    # Cargar el CSV con chunksize para manejar archivos grandes
    print("Cargando archivo CSV...")
    df = pd.read_csv(csv_file_path, encoding="latin1")
    print(f"Archivo cargado exitosamente. Dimensiones: {df.shape}")
except Exception as e:
    print(f"Error al cargar el archivo: {e}")
    # Si el archivo es muy grande, usar chunks
    chunk_list = []
    for chunk in pd.read_csv(csv_file_path, chunksize=10000):
        chunk_list.append(chunk)
    df = pd.concat(chunk_list, ignore_index=True)
    print(f"Archivo cargado en chunks. Dimensiones: {df.shape}")

In [ ]:
# Mostrar y limpiar nombres de columnas para evitar KeyError
print("Nombres de columnas antes de limpiar:", df.columns.tolist())
df.columns = df.columns.str.strip()
print("Nombres de columnas después de limpiar:", df.columns.tolist())

In [ ]:
# Filtrar registros con laboratory nulo y product null
print("=== Filtrando registros nulos en laboratorio y producto ===")

# Verificar cantidad de registros con laboratory nulo antes del filtro
initial_count = len(df)
null_laboratory_count = df['laboratory'].isnull().sum()

print(f"Registros totales antes del filtro: {initial_count:,}")
print(f"Registros con laboratory nulo: {null_laboratory_count:,}")
print(f"Porcentaje de registros con laboratory nulo: {(null_laboratory_count/initial_count)*100:.2f}%")

# Filtrar registros donde laboratory no sea nulo
df = df[df['laboratory'].notna()].copy()


# Verificar resultados después del filtro
final_count = len(df)
remaining_nulls = df['laboratory'].isnull().sum()

print(f"\nRegistros después del filtro: {final_count:,}")
print(f"Registros eliminados: {initial_count - final_count:,}")
print(f"Registros con laboratory nulo restantes: {remaining_nulls}")

if remaining_nulls == 0:
    print("✅ Todos los registros con laboratory nulo han sido eliminados exitosamente")
else:
    print(f"⚠️  ALERTA: Aún quedan {remaining_nulls} registros con laboratory nulo")

print(f"\nNuevo tamaño del dataset: {df.shape}")
print(f"Reducción del dataset: {((initial_count - final_count) / initial_count) * 100:.2f}%")

# Ahora vemos el producto si hay nulos
initial_count = len(df)
null_product_count = df['product'].isnull().sum()

print(f"Registros totales antes del filtro: {initial_count:,}")
print(f"Registros con laboratory nulo: {null_product_count:,}")
print(f"Porcentaje de registros con laboratory nulo: {(null_product_count/initial_count)*100:.2f}%")

# Filtrar registros donde laboratory no sea nulo
df = df[df['product'].notna()].copy()


# Verificar resultados después del filtro
final_count = len(df)
remaining_nulls = df['product'].isnull().sum()

print(f"\nRegistros después del filtro: {final_count:,}")
print(f"Registros eliminados: {initial_count - final_count:,}")
print(f"Registros con product nulo restantes: {remaining_nulls}")

if remaining_nulls == 0:
    print("✅ Todos los registros con product nulo han sido eliminados exitosamente")
else:
    print(f"⚠️  ALERTA: Aún quedan {remaining_nulls} registros con product nulo")

print(f"\nNuevo tamaño del dataset: {df.shape}")
print(f"Reducción del dataset: {((initial_count - final_count) / initial_count) * 100:.2f}%")






In [ ]:
# Explorar la estructura de los datos
print("=== INFORMACIÓN GENERAL DEL DATASET ===")
print(f"Número de filas: {len(df):,}")
print(f"Número de columnas: {len(df.columns)}")
print(f"\nColumnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\n=== PRIMERAS 5 FILAS ===")
display(df.head())

print("\n=== INFORMACIÓN DE TIPOS DE DATOS ===")
display(df.info())

print("\n=== ESTADÍSTICAS DESCRIPTIVAS ===")
display(df.describe())

In [ ]:
# Verificar valores únicos en las columnas principales
print("=== CONTEO DE VALORES ÚNICOS ===")
unique_counts = {
    'store': df['store'].nunique(),
    'supervisor': df['supervisor'].nunique(),
    'barcode': df['barcode'].nunique(),
    'product': df['product'].nunique(),
    'brand': df['brand'].nunique(),
    'laboratory': df['laboratory'].nunique(),
    'category': df['category'].nunique(),
    'type': df['type'].nunique(),
    'employee': df['employee'].nunique()
}

for column, count in unique_counts.items():
    print(f"{column}: {count:,} valores únicos")

print("\n=== VERIFICACIÓN DE VALORES NULOS ===")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0] if null_counts.sum() > 0 else "No hay valores nulos")

## 3. Crear Dataset de Stores (Tiendas)

Extraemos las tiendas únicas de la columna 'store' y creamos un dataset de stores con IDs únicos.

In [ ]:
# Crear dataset de stores
print("=== CREANDO DATASET DE STORES ===")
# Obtener combinaciones únicas de store, lat y lon
stores_unique = df[['store', 'lat', 'lon']].drop_duplicates().dropna(subset=['store'])

# Crear DataFrame de stores con IDs
stores_df = pd.DataFrame({
    'store_id': range(1, len(stores_unique) + 1),
    'store_name': stores_unique['store'].values,
    'store_lat': stores_unique['lat'].values,
    'store_lon': stores_unique['lon'].values
})

print(f"Se crearon {len(stores_df)} stores únicos con coordenadas")
display(stores_df.head(10))

print("\nÚltimos 5 stores:")
display(stores_df.tail())

# Crear diccionario para mapeo
store_mapping = dict(zip(stores_df['store_name'], stores_df['store_id']))
print(f"\nDiccionario de mapeo creado con {len(store_mapping)} elementos")

## 4. Crear Dataset de Products (Productos)

Extraemos los productos únicos de las columnas 'barcode' y 'product' y creamos un dataset de productos con IDs únicos.

In [ ]:
# Crear dataset de products
print("=== CREANDO DATASET DE PRODUCTS ===")

# Obtener combinaciones únicas de barcode y product
products_unique = df[['barcode', 'product']].drop_duplicates()
products_unique = products_unique.dropna()  # Remover filas con valores nulos

# Crear DataFrame de products con IDs
products_df = pd.DataFrame({
    'product_id': range(1, len(products_unique) + 1),
    'barcode': products_unique['barcode'].values,
    'product_name': products_unique['product'].values
})

print(f"Se crearon {len(products_df)} productos únicos")
print("\nPrimeros 10 productos:")
display(products_df.head(10))

print("\nÚltimos 5 productos:")
display(products_df.tail())

# Crear diccionario para mapeo usando barcode como clave
product_mapping = dict(zip(products_df['barcode'], products_df['product_id']))
print(f"\nDiccionario de mapeo creado con {len(product_mapping)} elementos")

## 5. Crear Dataset de Brands (Marcas)

Extraemos las marcas únicas de la columna 'brand' y creamos un dataset de marcas con IDs únicos.

In [ ]:
# Crear dataset de brands
print("=== CREANDO DATASET DE BRANDS ===")

# Obtener brands únicos
unique_brands = df['brand'].unique()
unique_brands = unique_brands[pd.notna(unique_brands)]  # Remover valores nulos si existen

# Crear DataFrame de brands con IDs
brands_df = pd.DataFrame({
    'brand_id': range(1, len(unique_brands) + 1),
    'brand_name': unique_brands
})

print(f"Se crearon {len(brands_df)} marcas únicas")
print("\nPrimeras 10 marcas:")
display(brands_df.head(10))

print("\nÚltimas 5 marcas:")
display(brands_df.tail())

# Crear diccionario para mapeo
brand_mapping = dict(zip(brands_df['brand_name'], brands_df['brand_id']))
print(f"\nDiccionario de mapeo creado con {len(brand_mapping)} elementos")

## 6. Crear Dataset de Laboratories (Laboratorios)

Extraemos los laboratorios únicos de la columna 'laboratory' y creamos un dataset de laboratorios con IDs únicos.

In [ ]:
# Crear dataset de laboratories
print("=== CREANDO DATASET DE LABORATORIES ===")

# Obtener laboratories únicos
unique_laboratories = df['laboratory'].unique()
unique_laboratories = unique_laboratories[pd.notna(unique_laboratories)]  # Remover valores nulos si existen

# Crear DataFrame de laboratories con IDs
laboratories_df = pd.DataFrame({
    'laboratory_id': range(1, len(unique_laboratories) + 1),
    'laboratory_name': unique_laboratories
})

print(f"Se crearon {len(laboratories_df)} laboratorios únicos")
print("\nPrimeros 10 laboratorios:")
display(laboratories_df.head(10))

print("\nÚltimos 5 laboratorios:")
display(laboratories_df.tail())

# Crear diccionario para mapeo
laboratory_mapping = dict(zip(laboratories_df['laboratory_name'], laboratories_df['laboratory_id']))
print(f"\nDiccionario de mapeo creado con {len(laboratory_mapping)} elementos")

## 7. Crear Dataset de Categorias:

Extraemos las categorias únicas de la columna 'category' y creamos un dataset de categorias con IDs únicos.

In [ ]:
# Crear dataset de groups
print("=== CREANDO DATASET DE category ===")

# Obtener groups únicos
unique_category = df['category'].unique()
unique_category = unique_category[pd.notna(unique_category)]  # Remover valores nulos si existen

# Crear DataFrame de category con IDs
category_df = pd.DataFrame({
    'category_id': range(1, len(unique_category) + 1),
    'category_name': unique_category
})

print(f"Se crearon {len(category_df)} category únicos")
print("\nTodos los grupos:")
display(category_df)

# Crear diccionario para mapeo
category_mapping = dict(zip(category_df['category_name'], category_df['category_id']))
print(f"\nDiccionario de mapeo creado con {len(category_mapping)} elementos")

## 8. Crear Dataset de Types (Tipos)

Extraemos los tipos únicos de la columna 'type' y creamos un dataset de tipos con IDs únicos.

In [ ]:
# Crear dataset de types
print("=== CREANDO DATASET DE TYPES ===")

# Obtener types únicos
unique_types = df['type'].unique()
unique_types = unique_types[pd.notna(unique_types)]  # Remover valores nulos si existen

# Crear DataFrame de types con IDs
types_df = pd.DataFrame({
    'type_id': range(1, len(unique_types) + 1),
    'type_name': unique_types
})

print(f"Se crearon {len(types_df)} tipos únicos")
print("\nTodos los tipos:")
display(types_df)

# Crear diccionario para mapeo
type_mapping = dict(zip(types_df['type_name'], types_df['type_id']))
print(f"\nDiccionario de mapeo creado con {len(type_mapping)} elementos")

## 9. Crear Dataset de Supervisores (Supervisorss)

Extraemos los tipos únicos de la columna 'supervisor' y creamos un dataset de tipos con IDs únicos.

In [ ]:
# Crear dataset de supervisor
print("=== CREANDO DATASET DE SUPERVISORS ===")

# Obtener supervisor únicos
unique_supervisor = df['supervisor'].unique()
unique_supervisor = unique_supervisor[pd.notna(unique_supervisor)]  # Remover valores nulos si existen

# Crear DataFrame de types con IDs
supervisor_df = pd.DataFrame({
    'supervisor_id': range(1, len(unique_supervisor) + 1),
    'supervisor_name': unique_supervisor
})

print(f"Se crearon {len(supervisor_df)} supervisor únicos")
print("\nTodos los supervisor:")
display(supervisor_df)

# Crear diccionario para mapeo
supervisor_mapping = dict(zip(supervisor_df['supervisor_name'], supervisor_df['supervisor_id']))
print(f"\nDiccionario de mapeo creado con {len(supervisor_mapping)} elementos")

## 10. Crear Dataset de Empleados (Employees)

Extraemos los tipos únicos de la columna 'employee' y creamos un dataset de tipos con IDs únicos.

In [ ]:
# Crear dataset de empleados
print("=== CREANDO DATASET DE Empleados ===")

# Obtener supervisor únicos
unique_employee = df['employee'].unique()
unique_employee = unique_employee[pd.notna(unique_employee)]  # Remover valores nulos si existen

# Crear DataFrame de types con IDs
employee_df = pd.DataFrame({
    'employee_id': range(1, len(unique_employee) + 1),
    'employee_name': unique_employee
})

print(f"Se crearon {len(employee_df)} employee únicos")
print("\nTodos los supervisor:")
display(employee_df)

# Crear diccionario para mapeo
employee_mapping = dict(zip(employee_df['employee_name'], employee_df['employee_id']))
print(f"\nDiccionario de mapeo creado con {len(employee_mapping)} elementos")

## 10. Establecer Relaciones Entre Entidades

Creamos las relaciones de clave foránea mapeando el dataset original a los IDs de las entidades recién creadas.

In [ ]:
# Crear dataset principal con foreign keys
print("=== ESTABLECIENDO RELACIONES ENTRE ENTIDADES ===")

# Crear una copia del dataset original
main_df = df.copy()

# Mapear los IDs de las entidades
print("Mapeando store_id...")
main_df['store_id'] = main_df['store'].map(store_mapping)

# Mapear los IDs de las entidades
print("Mapeando supervisor_id...")
main_df['supervisor_id'] = main_df['supervisor'].map(supervisor_mapping)

print("Mapeando product_id...")
main_df['product_id'] = main_df['barcode'].map(product_mapping)

print("Mapeando brand_id...")
main_df['brand_id'] = main_df['brand'].map(brand_mapping)

print("Mapeando laboratory_id...")
main_df['laboratory_id'] = main_df['laboratory'].map(laboratory_mapping)

print("Mapeando category_id...")
main_df['category_id'] = main_df['category'].map(category_mapping)

print("Mapeando type_id...")
main_df['type_id'] = main_df['type'].map(type_mapping)


print("Mapeando employee_id...")
main_df['employee_id'] = main_df['employee'].map(employee_mapping)

# Reordenar columnas


columns_order = [
    'year','month','day','time',
    'product_id','brand_id','laboratory_id','category_id','type_id','quantity','total','ticket','employee_id',
    'physician','supervisor_id','store_id',
    'year','month','day','time','product','brand','laboratory','category','type','quantity','total',
    'ticket','employee','physician','supervisor','store']

main_df = main_df[columns_order]

print("\n=== DATASET PRINCIPAL CON FOREIGN KEYS ===")
print(f"Dimensiones: {main_df.shape}")
print("\nPrimeras 5 filas:")
display(main_df.head())

In [ ]:
# Verificar columnas antes de reordenar para evitar KeyError
print("Columnas actuales en main_df:", main_df.columns.tolist())
print("Columnas requeridas:", columns_order)
missing_cols = [col for col in columns_order if col not in main_df.columns]
if missing_cols:
    print("Columnas faltantes:", missing_cols)
else:
    print("Todas las columnas requeridas están presentes.")

## 11. Validar Integridad de Datos

Verificamos la consistencia de los datos y validamos que todas las relaciones estén correctamente establecidas.

In [ ]:
# Validación de integridad de datos
print("=== VALIDACIÓN DE INTEGRIDAD DE DATOS ===")

# Verificar que no hay valores nulos en los IDs
id_columns = ['store_id', 'supervisor_id', 'product_id', 'brand_id', 'laboratory_id', 'category_id', 'type_id',
              'employee_id']

print("\n1. Verificando valores nulos en IDs:")
null_counts = main_df[id_columns].isnull().sum()
for col in id_columns:
    null_count = null_counts[col]
    print(f"   {col}: {null_count} valores nulos")
    if null_count > 0:
        print(f"   ⚠️  ALERTA: Hay {null_count} valores nulos en {col}")

# Verificar rangos de IDs
print("\n2. Verificando rangos de IDs:")
validations = [
    ('store_id', len(stores_df)),
    ('supervisor_id', len(supervisor_df)),
    ('product_id', len(products_df)),
    ('brand_id', len(brands_df)),
    ('laboratory_id', len(laboratories_df)),
    ('category_id', len(category_df)),
    ('type_id', len(types_df)),
    ('employee_id', len(employee_df))
]

for id_col, expected_max in validations:
    actual_max = main_df[id_col].max()
    actual_min = main_df[id_col].min()
    print(f"   {id_col}: rango [{actual_min}, {actual_max}], esperado [1, {expected_max}]")
    if actual_max > expected_max or actual_min < 1:
        print(f"   ⚠️  ALERTA: Rango fuera de lo esperado en {id_col}")

# Verificar consistencia de mapeos
print("\n3. Verificando consistencia de mapeos:")
mapping_checks = [
    ('store', 'store_id', store_mapping),
    ('supervisor', 'supervisor_id', supervisor_mapping),
    ('brand', 'brand_id', brand_mapping),
    ('laboratory', 'laboratory_id', laboratory_mapping),
    ('category', 'category_id', category_mapping),
    ('type', 'type_id', type_mapping),
    ('employee', 'employee_id', employee_mapping)
]

for original_col, id_col, mapping_dict in mapping_checks:
    sample_size = min(1000, len(main_df))
    sample_df = main_df.sample(n=sample_size, random_state=42)
    
    consistent = True
    for idx, row in sample_df.iterrows():
        original_value = row[original_col]
        mapped_id = row[id_col]
        expected_id = mapping_dict.get(original_value)
        
        if mapped_id != expected_id:
            consistent = False
            break
    
    status = "✅ CORRECTO" if consistent else "❌ ERROR"
    print(f"   {original_col} -> {id_col}: {status}")

print("\n4. Resumen de validación:")
total_nulls = main_df[id_columns].isnull().sum().sum()
print(f"   - Total de valores nulos en IDs: {total_nulls}")
print(f"   - Total de filas en dataset principal: {len(main_df):,}")
print(f"   - Porcentaje de integridad: {((len(main_df) * len(id_columns) - total_nulls) / (len(main_df) * len(id_columns))) * 100:.2f}%")

## 11. Exportar Datasets de Entidades

Guardamos todos los datasets de entidades creados y el dataset principal con claves foráneas en archivos CSV separados.

In [ ]:
# Exportar todos los datasets
print("=== EXPORTANDO DATASETS DE ENTIDADES ===")

# Crear directorio para los archivos exportados si no existe
output_dir = 'resultado_modelo_entidad_relacion'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Directorio '{output_dir}' creado")

# Lista de datasets a exportar
datasets_to_export = [
    (stores_df, 'stores.csv', 'Stores'),
    (supervisor_df, 'supervisors.csv', 'Supervisors'),
    (products_df, 'products.csv', 'Products'),
    (brands_df, 'brands.csv', 'Brands'),
    (laboratories_df, 'laboratories.csv', 'Laboratories'),
    (category_df, 'categories.csv', 'Categories'),
    (types_df, 'types.csv', 'Types'),
    (employee_df, 'employees.csv', 'Employees')
]

# Exportar datasets de entidades
print("\nExportando datasets de entidades:")
for dataset, filename, name in datasets_to_export:
    filepath = os.path.join(output_dir, filename)
    dataset.to_csv(filepath, index=False, encoding='utf-8')
    print(f"   ✅ {name}: {filepath} ({len(dataset):,} filas)")

# Exportar dataset principal con foreign keys
#print("\nExportando dataset principal:")
#main_filepath = os.path.join(output_dir, 'main_dataset_with_foreign_keys2.csv')
#main_df.to_csv(main_filepath, index=False, encoding='utf-8')
#print(f"   ✅ Dataset principal: {main_filepath} ({len(main_df):,} filas)")

# Crear dataset principal solo con IDs y métricas
main_clean_df = main_df[['year', 'month','day','time', 'store_id','supervisor_id', 'product_id', 
                        'brand_id', 'laboratory_id', 'category_id', 'type_id', 
                        'quantity', 'total','employee_id']].copy()

main_clean_filepath = os.path.join(output_dir, 'sales.csv')
main_clean_df.to_csv(main_clean_filepath, index=False, encoding='utf-8')
print(f"   ✅ Dataset principal normalizado: {main_clean_filepath} ({len(main_clean_df):,} filas)")

print(f"\n🎉 Todos los archivos han sido exportados exitosamente al directorio '{output_dir}'")

In [ ]:
# Resumen final del modelo entidad-relación
print("=" * 60)
print("         RESUMEN DEL MODELO ENTIDAD-RELACIÓN")
print("=" * 60)

print("\n📊 ENTIDADES CREADAS:")
entities_summary = [
    ("Stores", len(stores_df), "Tiendas/Sucursales"),
    ("Supervisors", len(supervisor_df), "Supervisores"),
    ("Products", len(products_df), "Productos (barcode + nombre)"),
    ("Brands", len(brands_df), "Marcas"),
    ("Laboratories", len(laboratories_df), "Laboratorios"),
    ("Category", len(category_df), "Categorías de productos"),
    ("Types", len(types_df), "Tipos de productos"),
    ("Employees", len(employee_df), "Empleados")
]

for entity, count, description in entities_summary:
    print(f"   📋 {entity:<12}: {count:>6,} registros ({description})")

print(f"\n📈 DATASET PRINCIPAL:")
print(f"   📊 Registros totales: {len(main_df):,}")
print(f"   🔗 Relaciones establecidas: {len(id_columns)}")
print(f"   ✅ Integridad de datos: {((len(main_df) * len(id_columns) - total_nulls) / (len(main_df) * len(id_columns))) * 100:.2f}%")

print(f"\n💾 ARCHIVOS GENERADOS:")
print(f"   📁 Directorio: {output_dir}/")
print(f"   📄 Entidades: 6 archivos CSV")
print(f"   📄 Dataset principal: 2 archivos CSV")
print(f"   📄 Total de archivos: 8")

print(f"\n🔍 ESTRUCTURA DEL MODELO:")
print(f"   • Cada entidad tiene su propio ID único (1, 2, 3, ...)")
print(f"   • El dataset principal contiene foreign keys a todas las entidades")
print(f"   • Se mantienen los datos originales para referencia")
print(f"   • Se crea una versión normalizada solo con IDs y métricas")

print("\n" + "=" * 60)
print("✅ MODELO ENTIDAD-RELACIÓN COMPLETADO EXITOSAMENTE")
print("=" * 60)